# 00 Setup and First Capture

This notebook verifies the first things a PyTorch user needs: TorchLens imports, `tl.trace` runs on a tiny model, and the returned `Trace` can summarize what happened. A `Trace` is the top-level audit object for one model execution.

We start with a small CPU-only module so the notebook is fast and deterministic. The model has two modules and one functional op, which is enough to make a useful graph without noise.

## API Status

This series runs against the current TorchLens checkout. Known glossary-vs-build gaps used in later notebooks:

| Topic | Glossary/target name | Current build used here |
|---|---|---|
| Quantity classes | `tl.Quantity` / `tl.Bytes` / `tl.Duration` / `tl.Flops` / `tl.Macs` | Exported; memory/duration/flops/macs values format themselves with `str(...)`. |
| Activation memory | `activation_memory` | Current records expose quantity fields such as `activation_memory` and aggregate fields such as `total_activation_memory`. |
| Site lookup | `tl.find_sites(...)` | Use `Trace.find_sites(...)`. |
| Export | canonical package exporters | Use `torchlens.export.csv/json/parquet(log, path)`. |


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

In [ ]:
import torch
from torch import nn
import torchlens as tl

torch.manual_seed(7)


class TinyClassifier(nn.Module):
    """Tiny MLP used for the first TorchLens audit capture."""

    def __init__(self) -> None:
        """Initialize the layers."""
        super().__init__()
        self.hidden = nn.Linear(4, 6)
        self.output = nn.Linear(6, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run one forward pass."""
        hidden = torch.relu(self.hidden(x))
        return self.output(hidden)


model = TinyClassifier().eval()
x = torch.randn(3, 4)
print(f"torchlens version: {tl.__version__}")
print(f"input shape: {tuple(x.shape)}")

`tl.trace(model, x)` runs the model once and records tensor-producing operations. By default this small trace saves activations, so we can inspect both metadata and tensors.

In [ ]:
trace = tl.trace(model, x)
print(type(trace).__name__)
print(f"model class: {trace.model_class_name}")
print(f"number of indexed records: {len(trace)}")
print(f"input layers: {trace.input_layers}")
print(f"output layers: {trace.output_layers}")
print(f"total activation memory: {trace.total_activation_memory}")

The `Trace` representation is intentionally compact: it lists the layer order and tells you whether activations were saved. This is the quickest “it worked” check.

In [ ]:
print(trace)

`summary()` gives a table-oriented overview for a user-facing report. Here we show the default summary and then a smaller custom field selection.

In [ ]:
print(trace.summary())
print("\nSelected fields only:\n")
print(trace.summary(fields=["name", "shape", "params"]))

A trace creates an explicit output layer with the model result. In this build, `raw_output` is only populated when raw-output saving is enabled, so the executable default check compares the saved output-layer tensor to a normal PyTorch forward pass.

In [ ]:
with torch.no_grad():
    expected = model(x)

output_layer = trace[trace.output_layers[0]]
print(f"raw_output saved by default: {trace.raw_output is not None}")
print(f"output layer: {output_layer.layer_label}")
print(f"output layer shape: {output_layer.shape}")
print(f"matches normal forward: {torch.allclose(output_layer.out, expected)}")

## Capture Knobs and Extraction Helpers
This section exercises the capture controls that exist in this build: selective saving via `layers_to_save`, input `transform`, `output_transform`, raw-save controls, disk-backed outputs, `module_filter`, `tl.peek`, `tl.extract`, `tl.batched_extract`, and `raise_on_nan`. `stop_after` is currently supported by `tl.peek`, while `tl.trace(..., stop_after=...)` raises `NotImplementedError`.

In [ ]:
import shutil

ARTIFACT_DIR = REPO_ROOT / "notebooks" / "audit" / "_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

hidden_label = next(layer.layer_label for layer in trace.layers if layer.func_name == "linear")
knob_trace = tl.trace(
    model,
    {"payload": x},
    transform=lambda payload: (payload["payload"],),
    layers_to_save=[hidden_label],
    activation_transform=lambda tensor: tensor.detach().mean(dim=0, keepdim=True),
    save_raw_activations=False,
    output_transform=lambda output: {"predicted_class": output.argmax(dim=-1).tolist()},
    save_raw_output=True,
)
knob_layer = knob_trace[hidden_label]
print(f"layers_to_save: {[hidden_label]}")
print(f"raw out retained for saved layer: {isinstance(knob_layer.out, torch.Tensor)}")
print(f"transformed_out shape: {tuple(knob_layer.transformed_out.shape)}")
print(f"raw_output from output_transform: {knob_trace.raw_output}")

disk_dir = ARTIFACT_DIR / "00_disk_outs"
if disk_dir.exists():
    shutil.rmtree(disk_dir)
disk_trace = tl.trace(
    model,
    x,
    layers_to_save="all",
    save_outs_to=disk_dir,
    keep_outs_in_memory=False,
)
disk_layer = disk_trace[hidden_label]
print(f"disk-backed out in memory: {isinstance(disk_layer.out, torch.Tensor)}")
print(f"disk artifact directory exists: {disk_dir.exists()}")
print(
    '> NOTE: selective disk streaming gap: save_outs_to requires layers_to_save="all" in this build.'
)

filtered = tl.trace(
    model,
    x,
    module_filter=lambda module: False,
)
print(
    "> NOTE: module_filter gap: filtering every module still leaves "
    f"{len(filtered.layers)} layers in this build ({trace.layer_labels} -> {filtered.layer_labels})."
)

if hasattr(tl, "peek"):
    peeked = tl.peek(model, x, hidden_label, stop_after=hidden_label)
    print(f"tl.peek({hidden_label!r}) shape: {tuple(peeked.shape)}")
else:
    print("> NOTE: tl.peek is not exported in this build.")

if hasattr(tl, "extract"):
    extracted = tl.extract(model, x, {"hidden": hidden_label})
    print(
        f"tl.extract keys/shapes: {[(key, tuple(value.shape)) for key, value in extracted.items()]}"
    )
else:
    print("> NOTE: tl.extract is not exported in this build.")

if hasattr(tl, "batched_extract"):
    batched = tl.batched_extract(
        model, torch.randn(4, 4), [hidden_label], batch_size=2, progress=False
    )
    print(f"tl.batched_extract shape: {tuple(batched[hidden_label].shape)}")
else:
    print("> NOTE: tl.batched_extract is not exported in this build.")

try:
    tl.trace(model, x, stop_after=hidden_label)
except NotImplementedError as exc:
    print(f"> NOTE: tl.trace stop_after gap: {exc}")


class NanNet(nn.Module):
    """Model that intentionally creates a NaN for failure-mode capture."""

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        """Return value divided by itself."""
        return value / value


try:
    tl.trace(NanNet().eval(), torch.tensor([0.0]), raise_on_nan=True)
except Exception as exc:
    print(f"raise_on_nan caught: {type(exc).__name__}: {str(exc).splitlines()[0]}")

## 🔧 Sandbox

Mini-experiment: change `hidden_width`, `batch_size`, and `activation_kind`. Expected observation: wider/bigger models add parameters or memory, while the output shape follows `batch_size`. The cell prints a before/after delta.

In [ ]:
hidden_width = 5
batch_size = 2
activation_kind = "tanh"

activation = nn.Tanh() if activation_kind == "tanh" else nn.ReLU()
sandbox_model = nn.Sequential(
    nn.Linear(4, hidden_width),
    activation,
    nn.Linear(hidden_width, 1),
).eval()
sandbox_x = torch.randn(batch_size, 4)
sandbox_trace = tl.trace(sandbox_model, sandbox_x)

print(f"baseline layers: {len(trace.layers)}, sandbox layers: {len(sandbox_trace.layers)}")
print(f"baseline memory: {trace.total_activation_memory}")
print(f"sandbox memory:  {sandbox_trace.total_activation_memory}")
print(
    f"output shape changed: {trace[trace.output_layers[0]].shape} -> {sandbox_trace[sandbox_trace.output_layers[0]].shape}"
)
baseline_params = sum(layer.num_params for layer in trace.layers if layer.num_params is not None)
sandbox_params = sum(
    layer.num_params for layer in sandbox_trace.layers if layer.num_params is not None
)
print(f"parameter delta: {sandbox_params - baseline_params}")